# Set-up

In [1]:
import os
import glob

import pandas as pd
import snapatac2 as snap

from tqdm.auto import tqdm

The history saving thread hit an unexpected error (DatabaseError('database disk image is malformed')).History will not be written to the database.


/cellar/users/aklie/opt/miniconda3/envs/scverse-lite-py311/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# Input data paths (TODO: modify these paths to point to a directory where each group has it's own file)
path_fragments = "/cellar/users/aklie/data/datasets/sc-islet-differentiation_10X-Multiome/processed"  # Directory containing fragment files
path_sample_metadata = "/cellar/users/aklie/data/datasets/sc-islet-differentiation_10X-Multiome/results/1_get_data/sample_metadata.tsv"
path_cell_metadata = "/cellar/users/aklie/data/datasets/sc-islet-differentiation_10X-Multiome/results/1_get_data/cell_metadata.tsv"  # Path to cell metadata CSV file

path_out = "/cellar/users/aklie/data/datasets/sc-islet-differentiation_10X-Multiome/results/2_process_data"

In [3]:
# Load sample metadata
sample_metadata = pd.read_csv(path_sample_metadata, sep="\t")
sample_metadata

,sample,cellranger_folder,batch,stage,day,rep,path_frag_file
0,HZ015_D4_rep1,HZ184_MM1008_MM1009_H1_S1_rep1,batch1,D4,4,1,/cellar/users/aklie/data/datasets/sc-islet-dif...
1,HZ015_D4_rep2,HZ185_MM1010_MM1011_H1_S1_rep2,batch1,D4,4,2,/cellar/users/aklie/data/datasets/sc-islet-dif...
2,HZ015_D7_rep1,HZ189_MM1046_MM1053_H1_S2_rep2,batch1,D7,7,1,/cellar/users/aklie/data/datasets/sc-islet-dif...
3,XW002_D7_rep1,HZ_13_XW002_D7_rep1,batch2,D7,7,1,/cellar/users/aklie/data/datasets/sc-islet-dif...
4,HZ015_D9_rep1,HZ193_MM1050_MM1057_H1_S3_rep2,batch1,D9,9,1,/cellar/users/aklie/data/datasets/sc-islet-dif...
5,XW002_D9_rep1,HZ_14_XW002_D9_rep1,batch2,D9,9,1,/cellar/users/aklie/data/datasets/sc-islet-dif...
6,XW002_D12_rep1,HZ_15_XW002_D12_rep1,batch2,D12,12,1,/cellar/users/aklie/data/datasets/sc-islet-dif...
7,XW002_D12_rep2,HZ_16_XW002_D12_rep2,batch2,D12,12,2,/cellar/users/aklie/data/datasets/sc-islet-dif...
8,HZ021_D15_rep1,HZ_01_HZ021_D15_rep1,batch2,D15,15,1,/cellar/users/aklie/data/datasets/sc-islet-dif...
9,HZ021_D15_rep2,HZ_02_HZ021_D15_rep2,batch2,D15,15,2,/cellar/users/aklie/data/datasets/sc-islet-dif...


In [4]:
# Load cell metadata
cell_metadata = pd.read_csv(path_cell_metadata, sep="\t")
cell_metadata.head()

/tmp/ipykernel_1453682/2871173399.py:2: DtypeWarning: Columns (61,62,63,64,65,66) have mixed types. Specify dtype option on import or set low_memory=False.
  cell_metadata = pd.read_csv(path_cell_metadata, sep="\t")


,Unnamed: 0,orig.ident,nCount_RNA,nFeature_RNA,gex_barcode_cellranger,atac_barcode_cellranger,is_cell_cellranger,excluded_reason_cellranger,gex_raw_reads_cellranger,gex_mapped_reads_cellranger,...,reg.cca.clusters,seurat_clusters,sct.cca.clusters,reg.rpca.clusters,sct.rpca.clusters,reg.harmony.clusters,sct.harmony.clusters,mnn.clusters,rna_annotation,sample_name
0,rna_HZ015_D4_rep1_dual_modality_QCed_single_ti...,SeuratProject,2694,1778,CCAGCTAAGAGAGCCG-1,CGCTAACTCTAGGAAC-1,1,0,7833,7363,...,0,9,0,2,1,0,9,1,DE,HZ015_D4_rep1
1,rna_HZ015_D4_rep1_dual_modality_QCed_single_ti...,SeuratProject,2175,1404,GTCATTAAGTAAGAAC-1,CGTGTGTTCCTTCAAG-1,1,0,5790,5228,...,0,0,4,0,1,0,0,14,DE,HZ015_D4_rep1
2,rna_HZ015_D4_rep1_dual_modality_QCed_single_ti...,SeuratProject,1034,754,ATCCTGACAAGGTAAC-1,GATAACGGTTAGCTAC-1,1,0,2999,2754,...,0,1,2,0,1,0,1,5,DE,HZ015_D4_rep1
3,rna_HZ015_D4_rep1_dual_modality_QCed_single_ti...,SeuratProject,4914,2325,CAGCCAATCCGGTATG-1,GTCATTTAGTTAACGA-1,1,0,13873,12810,...,0,6,12,13,15,0,6,6,DE,HZ015_D4_rep1
4,rna_HZ015_D4_rep1_dual_modality_QCed_single_ti...,SeuratProject,5323,2408,CCTGTAACAGCTCATA-1,GTTAGGAGTTCAAGCT-1,1,0,15315,14466,...,0,3,7,2,5,0,3,4,DE,HZ015_D4_rep1


In [5]:
# Create barcode to cell id mapping
cell_labels = dict(zip(cell_metadata['sample_name'] + "#" + cell_metadata['gex_barcode_cellranger'], cell_metadata['rna_annotation']))
print(cell_metadata['rna_annotation'].value_counts())

rna_annotation
DE                         13130
early_SC_beta              10886
PGT1                       10414
PFG2                       10164
early_SC_EC                 8441
early_ENP                   7572
PP2                         6789
PP1                         5062
late_SC_beta                4937
late_SC_alpha               4630
late_SC_EC                  4343
early_SC_alpha              3274
PFG1                        2693
exocrine                    2297
PGT2                        1895
liver                       1774
PGT3                        1653
ENP_phase1                  1622
late_ENP                     883
SC_delta_GHRL                856
proliferating_endocrine      454
FB_FLT1                      311
Name: count, dtype: int64


In [6]:
# Get a dictionary of fragments files for each group of interest (TODO: modify the extension if necessary)
fragments_dict = sample_metadata.set_index('sample')["path_frag_file"].to_dict()
fragments_dict

{'HZ015_D4_rep1': '/cellar/users/aklie/data/datasets/sc-islet-differentiation_10X-Multiome/processed/batch1/HZ184_MM1008_MM1009_H1_S1_rep1/outs/atac_fragments.tsv.gz',
 'HZ015_D4_rep2': '/cellar/users/aklie/data/datasets/sc-islet-differentiation_10X-Multiome/processed/batch1/HZ185_MM1010_MM1011_H1_S1_rep2/outs/atac_fragments.tsv.gz',
 'HZ015_D7_rep1': '/cellar/users/aklie/data/datasets/sc-islet-differentiation_10X-Multiome/processed/batch1/HZ189_MM1046_MM1053_H1_S2_rep2/outs/atac_fragments.tsv.gz',
 'XW002_D7_rep1': '/cellar/users/aklie/data/datasets/sc-islet-differentiation_10X-Multiome/processed/batch2/HZ_13_XW002_D7_rep1/outs/atac_fragments.tsv.gz',
 'HZ015_D9_rep1': '/cellar/users/aklie/data/datasets/sc-islet-differentiation_10X-Multiome/processed/batch1/HZ193_MM1050_MM1057_H1_S3_rep2/outs/atac_fragments.tsv.gz',
 'XW002_D9_rep1': '/cellar/users/aklie/data/datasets/sc-islet-differentiation_10X-Multiome/processed/batch2/HZ_14_XW002_D9_rep1/outs/atac_fragments.tsv.gz',
 'XW002_D12_re

In [7]:
# Subset to first 2
sub_files_dict = {k: fragments_dict[k] for k in list(fragments_dict.keys())[:2]}
sub_files_dict

{'HZ015_D4_rep1': '/cellar/users/aklie/data/datasets/sc-islet-differentiation_10X-Multiome/processed/batch1/HZ184_MM1008_MM1009_H1_S1_rep1/outs/atac_fragments.tsv.gz',
 'HZ015_D4_rep2': '/cellar/users/aklie/data/datasets/sc-islet-differentiation_10X-Multiome/processed/batch1/HZ185_MM1010_MM1011_H1_S1_rep2/outs/atac_fragments.tsv.gz'}

# Import fragments

In [ ]:
def mk_output_name(name):
    return path_out + "/adatas/" + name + ".h5ad"

def import_fragments(files_dict):
    files = list(files_dict.values())
    cts = list(files_dict.keys())
    names = [mk_output_name(ct) for ct in cts]
    print(names)
    adatas = snap.pp.import_fragments(
        files,
        chrom_sizes=snap.genome.hg38,
        file=names,
        sorted_by_barcode=False,
        chunk_size=1000000,
        
    )
    for adata in adatas:
        adata.close()

In [21]:
os.makedirs(path_out + "/adatas", exist_ok=True)

In [22]:
import_fragments(sub_files_dict)

['/cellar/users/aklie/data/datasets/sc-islet-differentiation_10X-Multiome/results/2_process_data/adatas/HZ015_D4_rep1.h5ad', '/cellar/users/aklie/data/datasets/sc-islet-differentiation_10X-Multiome/results/2_process_data/adatas/HZ015_D4_rep2.h5ad']


100%|██████████| 2/2 [1:16:57<00:00, 2308.69s/it]  


In [23]:
# Load in each anndata one by one prepend sample# to barcodes and sample info and save
def add_sample_to_barcode():
    files = glob.glob(path_out + "/adatas/*.h5ad")
    for file in tqdm(files):
        sample = file.split('/')[-1].split('.h5ad')[0]
        print(f"Processing {sample}")
        adata = snap.read(file, backed=None)
        obs_name = [sample + "#" + bc for bc in adata.obs_names]
        adata.obs.index = obs_name
        adata.write_h5ad(file)

In [24]:
add_sample_to_barcode()

  0%|          | 0/2 [00:00<?, ?it/s]

Processing HZ015_D4_rep1


 50%|█████     | 1/2 [01:24<01:24, 84.37s/it]

Processing HZ015_D4_rep2


100%|██████████| 2/2 [02:16<00:00, 68.31s/it]


In [25]:
file = "/cellar/users/aklie/data/datasets/sc-islet-differentiation_10X-Multiome/results/2_process_data/adatas/HZ015_D4_rep1.h5ad"
adata = snap.read(file, backed=None)
adata

/cellar/users/aklie/opt/miniconda3/envs/scverse-lite-py311/lib/python3.11/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


AnnData object with n_obs × n_vars = 16936 × 0
    obs: 'n_fragment', 'frac_dup', 'frac_mito'
    uns: 'reference_sequences'
    obsm: 'fragment_paired'

In [27]:
# check to see if barcodes are annotated
adata.obs.index.map(cell_labels).value_counts()

DE         6070
PGT3        284
PGT1        128
PGT2         96
FB_FLT1       7
PFG1          5
Name: count, dtype: int64

# Make AnnDataset

In [30]:
def create_dataset():
    files = glob.glob("/cellar/users/aklie/data/datasets/sc-islet-differentiation_10X-Multiome/results/2_process_data/adatas/*.h5ad")
    dataset = snap.AnnDataSet(
        adatas=[(fl.split('/')[-1].split('.h5ad')[0], fl) for fl in files],
        filename='/cellar/users/aklie/data/datasets/sc-islet-differentiation_10X-Multiome/results/2_process_data/dataset.h5ads',
        add_key="sample",
    )
    dataset, _ = dataset.subset(obs_indices=[x for x in dataset.obs_names if x in cell_labels], out='/cellar/users/aklie/data/datasets/sc-islet-differentiation_10X-Multiome/results/2_process_data/subset')
    dataset.obs['cell_type'] = [cell_labels[x] for x in dataset.obs_names]
    dataset.close()

In [31]:
create_dataset()

In [8]:
dataset = snap.read_dataset(path_out + '/subset/_dataset.h5ads')
dataset

AnnDataSet object with n_obs x n_vars = 104080 x 0 backed at '/cellar/users/aklie/data/datasets/sc-islet-differentiation_10X-Multiome/results/2_process_data/subset/_dataset.h5ads'
contains 14 AnnData objects with keys: 'HZ015_D4_rep1', 'HZ015_D4_rep2', 'HZ015_D7_rep1', 'HZ015_D9_rep1', 'HZ019_D45_rep1', 'HZ019_D45_rep2', 'HZ021_D15_rep1', 'HZ021_D15_rep2', 'HZ021_D22_rep1', 'HZ021_D22_rep2', 'XW002_D12_rep1', 'XW002_D12_rep2', 'XW002_D7_rep1', 'XW002_D9_rep1'
    obs: 'sample', 'cell_type'
    uns: 'AnnDataSet', 'reference_sequences'

In [9]:
dataset.close()

# Subset to endocrine and non-endocrine

In [8]:
endocrine_cell_types = [
    "early_SC_beta",
    "late_SC_beta",
    "early_SC_alpha",
    "late_SC_alpha",
    "early_SC_EC",
    "late_SC_EC",
    "SC_delta_GHRL",
    "proliferating_endocrine",
    "early_ENP",
    "late_ENP",
    "ENP_phase1",
]

non_endocrine_cell_types = [
    "DE",
    "PGT1",
    "PGT2",
    "PGT3",
    "PFG1",
    "PFG2",
    "PP1",
    "PP2",
    "exocrine",
    "liver",
    "FB_FLT1",
]

In [11]:
# Load the full subset dataset
dataset = snap.read_dataset(path_out + '/subset/_dataset.h5ads')
print(dataset)
print(f"Total cells: {dataset.n_obs}")

AnnDataSet object with n_obs x n_vars = 104080 x 0 backed at '/cellar/users/aklie/data/datasets/sc-islet-differentiation_10X-Multiome/results/2_process_data/subset/_dataset.h5ads'
contains 14 AnnData objects with keys: 'HZ015_D4_rep1', 'HZ015_D4_rep2', 'HZ015_D7_rep1', 'HZ015_D9_rep1', 'HZ019_D45_rep1', 'HZ019_D45_rep2', 'HZ021_D15_rep1', 'HZ021_D15_rep2', 'HZ021_D22_rep1', 'HZ021_D22_rep2', 'XW002_D12_rep1', 'XW002_D12_rep2', 'XW002_D7_rep1', 'XW002_D9_rep1'
    obs: 'sample', 'cell_type', 'sample_name', 'replicate', 'diff_batch', 'diff_stage'
    uns: 'AnnDataSet', 'reference_sequences', 'macs3', 'macs3_replicate'
Total cells: 104080


In [12]:
# Subset to endocrine cell types
endocrine_indices = [i for i, ct in enumerate(dataset.obs['cell_type']) if ct in endocrine_cell_types]
print(f"Endocrine cells: {len(endocrine_indices)}")
endocrine_dataset, _ = dataset.subset(obs_indices=endocrine_indices, out=path_out + '/endocrine')
endocrine_dataset.obs['cell_type'] = [dataset.obs['cell_type'][i] for i in endocrine_indices]
print(endocrine_dataset)
endocrine_dataset.close()

/tmp/ipykernel_1453682/2222388017.py:2: DeprecationWarning: `_import_from_c` is deprecated; use `_import_arrow_from_c` instead. If you are using an extension, please compile it with the latest 'pyo3-polars'
  endocrine_indices = [i for i, ct in enumerate(dataset.obs['cell_type']) if ct in endocrine_cell_types]


Endocrine cells: 47898


/tmp/ipykernel_1453682/2222388017.py:5: DeprecationWarning: `_import_from_c` is deprecated; use `_import_arrow_from_c` instead. If you are using an extension, please compile it with the latest 'pyo3-polars'
  endocrine_dataset.obs['cell_type'] = [dataset.obs['cell_type'][i] for i in endocrine_indices]


AnnDataSet object with n_obs x n_vars = 47898 x 0 backed at '/cellar/users/aklie/data/datasets/sc-islet-differentiation_10X-Multiome/results/2_process_data/endocrine/_dataset.h5ads'
contains 14 AnnData objects with keys: 'HZ015_D4_rep1', 'HZ015_D4_rep2', 'HZ015_D7_rep1', 'HZ015_D9_rep1', 'HZ019_D45_rep1', 'HZ019_D45_rep2', 'HZ021_D15_rep1', 'HZ021_D15_rep2', 'HZ021_D22_rep1', 'HZ021_D22_rep2', 'XW002_D12_rep1', 'XW002_D12_rep2', 'XW002_D7_rep1', 'XW002_D9_rep1'
    obs: 'sample', 'cell_type', 'sample_name', 'replicate', 'diff_batch', 'diff_stage'
    uns: 'macs3_replicate', 'reference_sequences', 'macs3', 'AnnDataSet'


In [13]:
# Subset to non-endocrine cell types
non_endocrine_indices = [i for i, ct in enumerate(dataset.obs['cell_type']) if ct in non_endocrine_cell_types]
print(f"Non-endocrine cells: {len(non_endocrine_indices)}")
non_endocrine_dataset, _ = dataset.subset(obs_indices=non_endocrine_indices, out=path_out + '/non-endocrine')
non_endocrine_dataset.obs['cell_type'] = [dataset.obs['cell_type'][i] for i in non_endocrine_indices]
print(non_endocrine_dataset)
non_endocrine_dataset.close()

Non-endocrine cells: 56182


/tmp/ipykernel_1453682/3817187400.py:2: DeprecationWarning: `_import_from_c` is deprecated; use `_import_arrow_from_c` instead. If you are using an extension, please compile it with the latest 'pyo3-polars'
  non_endocrine_indices = [i for i, ct in enumerate(dataset.obs['cell_type']) if ct in non_endocrine_cell_types]
/tmp/ipykernel_1453682/3817187400.py:5: DeprecationWarning: `_import_from_c` is deprecated; use `_import_arrow_from_c` instead. If you are using an extension, please compile it with the latest 'pyo3-polars'
  non_endocrine_dataset.obs['cell_type'] = [dataset.obs['cell_type'][i] for i in non_endocrine_indices]


AnnDataSet object with n_obs x n_vars = 56182 x 0 backed at '/cellar/users/aklie/data/datasets/sc-islet-differentiation_10X-Multiome/results/2_process_data/non-endocrine/_dataset.h5ads'
contains 14 AnnData objects with keys: 'HZ015_D4_rep1', 'HZ015_D4_rep2', 'HZ015_D7_rep1', 'HZ015_D9_rep1', 'HZ019_D45_rep1', 'HZ019_D45_rep2', 'HZ021_D15_rep1', 'HZ021_D15_rep2', 'HZ021_D22_rep1', 'HZ021_D22_rep2', 'XW002_D12_rep1', 'XW002_D12_rep2', 'XW002_D7_rep1', 'XW002_D9_rep1'
    obs: 'sample', 'cell_type', 'sample_name', 'replicate', 'diff_batch', 'diff_stage'
    uns: 'macs3_replicate', 'AnnDataSet', 'macs3', 'reference_sequences'


In [14]:
# Close the full dataset
dataset.close()

# DONE!

---